Gold Layer - Binance Crypto Analytics & Aggregations

In [0]:
storage_account = "deproj1adls"
container = "crypto-data"

STORAGE_ACCOUNT_KEY = dbutils.secrets.get(scope="binance-secrets", key="storage-account-key")
spark.conf.set(f"fs.azure.account.key.{storage_account}.blob.core.windows.net", STORAGE_ACCOUNT_KEY)

SILVER_TABLE_PATH = f"wasbs://{container}@{storage_account}.blob.core.windows.net/silver/binance_trades_clean"
CHECKPOINT_PATH_HOURLY = f"wasbs://{container}@{storage_account}.blob.core.windows.net/checkpoints/gold_hourly_agg"
CHECKPOINT_PATH_DAILY = f"wasbs://{container}@{storage_account}.blob.core.windows.net/checkpoints/gold_daily_agg"
GOLD_HOURLY_PATH = f"wasbs://{container}@{storage_account}.blob.core.windows.net/gold/hourly_trading_metrics"
GOLD_DAILY_PATH = f"wasbs://{container}@{storage_account}.blob.core.windows.net/gold/daily_trading_summary"
TRIGGER_INTERVAL = "30 seconds"

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window

In [0]:
silver_stream = (
    spark.readStream
    .format("delta")
    .load(SILVER_TABLE_PATH)
)

silver_stream.printSchema()

## Hourly Trading Metrics

In [0]:
hourly_metrics = (
    silver_stream
    .withWatermark("trade_timestamp", "1 hour")
    .groupBy(
        window(col("trade_timestamp"), "1 hour"),
        col("symbol")
    )
    .agg(
        count("*").alias("trade_count"),
        sum("quantity").alias("total_quantity"),
        sum("trade_value_usd").alias("total_volume_usd"),
        avg("price").alias("avg_price"),
        min("price").alias("min_price"),
        max("price").alias("max_price"),
        first("price").alias("open_price"),
        last("price").alias("close_price"),
        sum(when(col("is_buyer_maker") == True, 1).otherwise(0)).alias("sell_trades"),
        sum(when(col("is_buyer_maker") == False, 1).otherwise(0)).alias("buy_trades"),
        sum(when(col("is_buyer_maker") == True, col("trade_value_usd")).otherwise(0)).alias("sell_volume_usd"),
        sum(when(col("is_buyer_maker") == False, col("trade_value_usd")).otherwise(0)).alias("buy_volume_usd")
    )
    .select(
        col("window.start").alias("hour_start"),
        col("window.end").alias("hour_end"),
        col("symbol"),
        col("trade_count"),
        round(col("total_quantity"), 8).alias("total_quantity"),
        round(col("total_volume_usd"), 2).alias("total_volume_usd"),
        round(col("avg_price"), 8).alias("avg_price"),
        round(col("min_price"), 8).alias("min_price"),
        round(col("max_price"), 8).alias("max_price"),
        round(col("open_price"), 8).alias("open_price"),
        round(col("close_price"), 8).alias("close_price"),
        round((col("close_price") - col("open_price")) / col("open_price") * 100, 2).alias("price_change_pct"),
        col("buy_trades"),
        col("sell_trades"),
        round(col("buy_volume_usd"), 2).alias("buy_volume_usd"),
        round(col("sell_volume_usd"), 2).alias("sell_volume_usd"),
        round((col("buy_volume_usd") - col("sell_volume_usd")) / col("total_volume_usd") * 100, 2).alias("buy_sell_ratio_pct"),
        current_timestamp().alias("gold_processing_time")
    )
)

In [0]:
hourly_query = (
    hourly_metrics
    .writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", CHECKPOINT_PATH_HOURLY)
    .trigger(processingTime=TRIGGER_INTERVAL)
    .start(GOLD_HOURLY_PATH)
)

print(f"Hourly aggregation stream started: {hourly_query.id}")

## Daily Trading Summary

In [0]:
daily_summary = (
    silver_stream
    .withWatermark("trade_timestamp", "24 hours")
    .groupBy(
        window(col("trade_timestamp"), "1 day"),
        col("symbol")
    )
    .agg(
        count("*").alias("trade_count"),
        sum("quantity").alias("total_quantity"),
        sum("trade_value_usd").alias("total_volume_usd"),
        avg("price").alias("avg_price"),
        min("price").alias("low_price"),
        max("price").alias("high_price"),
        first("price").alias("open_price"),
        last("price").alias("close_price"),
        stddev("price").alias("price_volatility"),
        approx_count_distinct("buyer_order_id").alias("unique_buyers"),
        approx_count_distinct("seller_order_id").alias("unique_sellers")
    )
    .select(
        col("window.start").alias("date_start"),
        to_date(col("window.start")).alias("trade_date"),
        col("symbol"),
        col("trade_count"),
        round(col("total_quantity"), 8).alias("total_quantity"),
        round(col("total_volume_usd"), 2).alias("total_volume_usd"),
        round(col("avg_price"), 8).alias("avg_price"),
        round(col("low_price"), 8).alias("low_price"),
        round(col("high_price"), 8).alias("high_price"),
        round(col("open_price"), 8).alias("open_price"),
        round(col("close_price"), 8).alias("close_price"),
        round((col("close_price") - col("open_price")) / col("open_price") * 100, 2).alias("daily_change_pct"),
        round((col("high_price") - col("low_price")) / col("low_price") * 100, 2).alias("daily_range_pct"),
        round(col("price_volatility"), 8).alias("price_volatility"),
        col("unique_buyers"),
        col("unique_sellers"),
        current_timestamp().alias("gold_processing_time")
    )
)

In [0]:
daily_query = (
    daily_summary
    .writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", CHECKPOINT_PATH_DAILY)
    .trigger(processingTime=TRIGGER_INTERVAL)
    .start(GOLD_DAILY_PATH)
)

print(f"Daily aggregation stream started: {daily_query.id}")

In [0]:
import time

print("Monitoring gold layer streams...\n")

for i in range(200):
    time.sleep(3)
    print(f"\nBatch {i+1}:")
    print(f"  Hourly: {hourly_query.status['message']}")
    print(f"  Daily: {daily_query.status['message']}")

In [0]:
hourly_df = spark.read.format("delta").load(GOLD_HOURLY_PATH)

print(f"Total hourly records: {hourly_df.count()}")
display(hourly_df.orderBy(col("hour_start").desc()).limit(20))

In [0]:
daily_df = spark.read.format("delta").load(GOLD_DAILY_PATH)

print(f"Total daily records: {daily_df.count()}")
display(daily_df.orderBy(col("trade_date").desc()).limit(10))

In [0]:
hourly_query.stop()
daily_query.stop()

## Analytics Queries

In [0]:
%sql
SELECT 
    symbol,
    hour_start,
    trade_count,
    total_volume_usd,
    open_price,
    close_price,
    price_change_pct,
    buy_sell_ratio_pct
FROM delta.`wasbs://crypto-data@deproj1adls.blob.core.windows.net/gold/hourly_trading_metrics`
WHERE hour_start >= current_timestamp() - INTERVAL 24 HOURS
ORDER BY hour_start DESC, total_volume_usd DESC
LIMIT 50

In [0]:
%sql
SELECT 
    symbol,
    trade_date,
    trade_count,
    total_volume_usd,
    open_price,
    close_price,
    high_price,
    low_price,
    daily_change_pct,
    price_volatility
FROM delta.`wasbs://crypto-data@deproj1adls.blob.core.windows.net/gold/daily_trading_summary`
ORDER BY trade_date DESC, total_volume_usd DESC
LIMIT 20

In [0]:
%sql
SELECT 
    symbol,
    ROUND(AVG(price_change_pct), 2) as avg_hourly_change,
    ROUND(MAX(price_change_pct), 2) as max_hourly_gain,
    ROUND(MIN(price_change_pct), 2) as max_hourly_loss,
    ROUND(SUM(total_volume_usd), 2) as total_volume_24h,
    SUM(trade_count) as total_trades_24h
FROM delta.`wasbs://crypto-data@deproj1adls.blob.core.windows.net/gold/hourly_trading_metrics`
WHERE hour_start >= current_timestamp() - INTERVAL 24 HOURS
GROUP BY symbol
ORDER BY total_volume_24h DESC

In [0]:
%sql
WITH ranked_hours AS (
    SELECT 
        symbol,
        hour_start,
        total_volume_usd,
        price_change_pct,
        ROW_NUMBER() OVER (PARTITION BY symbol ORDER BY total_volume_usd DESC) as volume_rank
    FROM delta.`wasbs://crypto-data@deproj1adls.blob.core.windows.net/gold/hourly_trading_metrics`
    WHERE hour_start >= current_timestamp() - INTERVAL 7 DAYS
)
SELECT 
    symbol,
    hour_start as peak_volume_hour,
    total_volume_usd,
    price_change_pct
FROM ranked_hours
WHERE volume_rank = 1
ORDER BY total_volume_usd DESC